# 04 — NFL Outcome Inventory & Hypothesis Discovery
## NFL Big Data Bowl 2027

**Purpose**:
1. Profile available rookie NFL outcome metrics from `player_play.csv`.
2. Link prospective Combine tracking features to first-season NFL performance.
3. Run exploratory correlation analysis across all feature-outcome pairs.
4. Export `data/processed/analysis_dataset.parquet` and `data/processed/correlation_summary.parquet`.

> **Temporal Integrity**: Combine features occur strictly pre-draft. NFL outcomes represent rookie season only.

In [ ]:
# ── Environment & Path Setup ────────────────────────────────────────────────
import sys
import os
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

for p in [Path("."), Path(".."), Path("/kaggle/working"), Path("/kaggle/working/kaggle_notebooks")]:
    if p.exists() and str(p.resolve()) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

try:
    from src.config import (
        ensure_dirs, FIGURES_DIR, TABLES_DIR, REPORTS_DIR, DATA_PROCESSED,
        COMBINE_FEATURES_PARQUET, PLAYER_MASTER_PARQUET, IN_KAGGLE, DATA_RAW
    )
    from src.data_loading import load_players, load_combine_results, load_player_play
    from src.player_mapping import assign_position_groups
    from src.combine_features import aggregate_player_features
    from src.nfl_features import aggregate_nfl_metrics
except ImportError:
    IN_KAGGLE = Path("/kaggle").exists()
    PROJECT_ROOT = Path("/kaggle/working") if IN_KAGGLE else Path(".").resolve()
    DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
    REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
    FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
    COMBINE_FEATURES_PARQUET = DATA_PROCESSED / "combine_features.parquet"
    PLAYER_MASTER_PARQUET = DATA_PROCESSED / "player_master.parquet"
    
    raw_candidates = [
        Path("/kaggle/input/nfl-big-data-bowl-2027"),
        Path("/kaggle/input/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"),
        Path("./data/raw/nfl-big-data-bowl-2027"),
        Path("./data/raw"),
    ]
    DATA_RAW = raw_candidates[0]
    for c in raw_candidates:
        if (c / "players.csv").exists():
            DATA_RAW = c
            break
    if not (DATA_RAW / "players.csv").exists() and Path("/kaggle/input").exists():
        found = list(Path("/kaggle/input").glob("**/players.csv"))
        if found:
            DATA_RAW = found[0].parent
            
    def ensure_dirs():
        DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
        REPORTS_DIR.mkdir(parents=True, exist_ok=True)
        FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    def load_players():
        return pd.read_csv(DATA_RAW / "players.csv")
    def load_player_play():
        return pd.read_csv(DATA_RAW / "player_play.csv")
        
    def assign_position_groups(df):
        pos_col = 'nfl_position' if 'nfl_position' in df.columns else ('position' if 'position' in df.columns else None)
        if not pos_col:
            return df
        mapping = {'WR': 'WR', 'TE': 'TE', 'CB': 'CB', 'FS': 'S', 'SS': 'S', 'S': 'S', 'DE': 'Edge', 'EDGE': 'Edge', 'OLB': 'Edge', 'ILB': 'LB', 'MLB': 'LB', 'LB': 'LB', 'DT': 'DT', 'NT': 'DT', 'T': 'OL', 'OT': 'OL', 'G': 'OL', 'OG': 'OL', 'C': 'OL', 'RB': 'RB'}
        df = df.copy()
        df['position_group'] = df[pos_col].map(mapping).fillna('Other')
        return df

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
ensure_dirs()
print(f"Environment: {'Kaggle' if Path('/kaggle').exists() else 'Local'}")
print(f"Data Directory: {DATA_RAW}")

## 1. Load Processed Combine Features & Player Play

In [ ]:
players = load_players()
players = assign_position_groups(players)
pos_map = players[['nfl_id', 'position_group']].drop_duplicates()

# Load combine features
if COMBINE_FEATURES_PARQUET.exists():
    features_df = pd.read_parquet(COMBINE_FEATURES_PARQUET)
else:
    print('combine_features.parquet not found. Creating placeholder features...')
    tracking = pd.read_csv(DATA_RAW / 'combine_tracking.csv')
    features_df = tracking.groupby(['nfl_id', 'drill_type', 'attempt']).agg(
        mean_speed=('s', 'mean'),
        max_speed=('s', 'max'),
    ).reset_index()
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    features_df.to_parquet(COMBINE_FEATURES_PARQUET, index=False)

try:
    from src.combine_features import aggregate_player_features
    player_features = aggregate_player_features(features_df, agg_strategy='best_attempt')
except Exception:
    player_features = features_df.sort_values('max_speed', ascending=False).drop_duplicates(['nfl_id', 'drill_type'])

player_features = player_features.merge(pos_map, on='nfl_id', how='left')
print(f'Combine features: {player_features.shape}')

player_play = load_player_play()
print(f'Player play rows: {len(player_play):,} | Unique NFL IDs: {player_play["nfl_id"].nunique():,}')

## 2. Aggregate Rookie NFL Outcomes

In [ ]:
# Aggregate rookie season outcome metrics per player
nfl_agg = player_play.groupby('nfl_id').agg(
    n_plays=('play_id', 'count'),
    mean_separation=('separation_at_pass_forward', 'mean'),
    mean_cushion=('cushion', 'mean'),
    mean_yac=('yards_after_catch', 'mean'),
    mean_get_off=('player_get_off', 'mean'),
    sack_allowed_rate=('sack_allowed', 'mean'),
    pressure_allowed_rate=('pressure_allowed', 'mean'),
    quick_pressure_rate=('quick_pressure', 'mean'),
).reset_index()

# Calculate YAC over expected if columns exist
if 'yards_after_catch' in player_play.columns and 'expected_yards_after_catch' in player_play.columns:
    player_play['yac_diff'] = player_play['yards_after_catch'] - player_play['expected_yards_after_catch']
    yac_oe = player_play.groupby('nfl_id')['yac_diff'].mean().reset_index(name='yac_over_expected')
    nfl_agg = nfl_agg.merge(yac_oe, on='nfl_id', how='left')

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
nfl_agg.to_parquet(DATA_PROCESSED / 'nfl_metrics.parquet', index=False)
print(f'Aggregated NFL outcomes saved: {nfl_agg.shape}')
display(nfl_agg.head())

## 3. Link Combine Features to NFL Outcomes (Analysis Dataset)

In [ ]:
# Pivot combine features to wide player format (feature_drill)
wide_feats = player_features.pivot_table(
    index=['nfl_id', 'position_group'],
    columns='drill_type',
    values=[c for c in player_features.columns if c not in ['nfl_id', 'position_group', 'drill_type', 'attempt']],
    aggfunc='first'
)
wide_feats.columns = [f'{col}_{drill}' for col, drill in wide_feats.columns]
wide_feats = wide_feats.reset_index()

# Merge with NFL outcomes
analysis_df = wide_feats.merge(nfl_agg, on='nfl_id', how='inner')
analysis_df.to_parquet(DATA_PROCESSED / 'analysis_dataset.parquet', index=False)
print(f'Analysis dataset generated: {analysis_df.shape} players')
display(analysis_df.head())

## 4. Hypothesis Discovery: Cross-Feature Spearman Correlations

In [ ]:
combine_cols = [c for c in wide_feats.columns if c not in ['nfl_id', 'position_group']]
nfl_cols = [c for c in nfl_agg.columns if c not in ['nfl_id', 'n_plays']]

results = []
for pos, grp in analysis_df.groupby('position_group'):
    if len(grp) < 10:
        continue
    for c_col in combine_cols:
        for n_col in nfl_cols:
            sub = grp[[c_col, n_col]].dropna()
            if len(sub) >= 10:
                r, p = stats.spearmanr(sub[c_col], sub[n_col])
                if not np.isnan(r):
                    results.append({
                        'position_group': pos,
                        'combine_feature': c_col,
                        'nfl_outcome': n_col,
                        'rho': float(r),
                        'p_value': float(p),
                        'n': len(sub)
                    })

df_corr = pd.DataFrame(results)
if not df_corr.empty:
    df_corr = df_corr.sort_values('p_value')
    df_corr.to_parquet(DATA_PROCESSED / 'correlation_summary.parquet', index=False)
    print(f'Calculated {len(df_corr)} correlation pairs.')
    print('\nTop 10 Exploratory Signals by unadjusted p-value:')
    display(df_corr.head(10))
else:
    print('No valid correlation pairs met sample size threshold.')

## 5. Export Hypothesis Candidates Report

In [ ]:
if not df_corr.empty:
    report_md = ["# NFL Big Data Bowl 2027 — Hypothesis Candidates Report",
                 f"*Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n",
                 "| Position | Combine Feature | NFL Outcome | Spearman rho | p-value | N |",
                 "|---|---|---|---|---|---|"]
    for _, row in df_corr.head(20).iterrows():
        report_md.append(f"| {row['position_group']} | {row['combine_feature']} | {row['nfl_outcome']} | {row['rho']:.3f} | {row['p_value']:.4f} | {row['n']} |")
    
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = REPORTS_DIR / 'hypothesis_candidates.md'
    out_path.write_text('\n'.join(report_md), encoding='utf-8')
    print(f'Report exported to: {out_path}')